# 03 — Model Evaluation & Ablation Study

Full evaluation of the trained BiLSTM+CNN model and ablation variants.

**Covers:**
- Per-class F1 scores
- Confusion matrix analysis
- ROC curves
- Ablation: LSTM-only vs CNN-only vs Hybrid
- Statistical comparison of variants

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import yaml
from sklearn.metrics import classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('../configs/train_config.yaml') as f:
    cfg = yaml.safe_load(f)

CLASS_NAMES = cfg['dataset']['class_names']
print(f'Device: {device} | Classes: {len(CLASS_NAMES)}')

In [ ]:
# ── Load evaluation results ────────────────────────────────────────────
metrics_path = '../results/metrics.json'

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)

    print('=== DETECTION MODEL — TEST RESULTS ===')
    print(f'  Accuracy   : {metrics["accuracy"]:.4f}')
    print(f'  Macro F1   : {metrics["macro_f1"]:.4f}')
    print(f'  ROC-AUC    : {metrics["roc_auc"]:.4f}')
    print(f'  Mean FPR   : {metrics["fpr"]:.4f}')
    print()
    print(metrics.get('classification_report_text', 'No classification report found'))
else:
    print('[INFO] No metrics found. Run: python scripts/evaluate.py')

In [ ]:
# ── Per-Class F1 Bar Chart ─────────────────────────────────────────────
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)

    f1_scores = metrics.get('per_class_f1', [])
    if f1_scores and len(f1_scores) == len(CLASS_NAMES):
        fig, ax = plt.subplots(figsize=(12, 5))
        colors = ['#1D9E75' if s >= 0.9 else '#F59E0B' if s >= 0.7 else '#EF4444'
                  for s in f1_scores]
        bars = ax.bar(CLASS_NAMES, f1_scores, color=colors, alpha=0.85)
        ax.axhline(0.9, color='gray', linestyle='--', alpha=0.5, label='F1=0.9 threshold')
        ax.set_ylim([0, 1.05])
        ax.set_ylabel('F1 Score')
        ax.set_title('Per-Class F1 Scores — Hybrid BiLSTM+CNN Model')
        ax.legend()
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.tight_layout()
        plt.savefig('../results/figures/per_class_f1.png', dpi=150, bbox_inches='tight')
        plt.show()

        worst = sorted(zip(CLASS_NAMES, f1_scores), key=lambda x: x[1])[:3]
        print('Hardest classes (lowest F1):',
              ', '.join(f'{n}: {s:.3f}' for n, s in worst))
    else:
        print('[INFO] Per-class F1 data not available in metrics.json')

In [ ]:
# ── Show Saved Confusion Matrix ────────────────────────────────────────
cm_path = '../results/figures/confusion_matrix.png'
if os.path.exists(cm_path):
    from IPython.display import Image
    display(Image(cm_path))
else:
    print('[INFO] Confusion matrix not yet generated.')
    print('Run: python scripts/evaluate.py')

In [ ]:
# ── Ablation Comparison ────────────────────────────────────────────────
ablation_path = '../results/ablation.json'

if os.path.exists(ablation_path):
    with open(ablation_path) as f:
        ablation = json.load(f)

    rows = []
    for variant, res in ablation.items():
        if 'status' not in res:
            rows.append({'Variant': variant, **res})

    if rows:
        df_abl = pd.DataFrame(rows).set_index('Variant')
        print('=== ABLATION STUDY RESULTS ===')
        print(df_abl.to_string())

        # Bar chart comparison
        metrics_to_plot = ['accuracy', 'macro_f1', 'roc_auc']
        available = [m for m in metrics_to_plot if m in df_abl.columns]

        if available:
            fig, ax = plt.subplots(figsize=(8, 4))
            df_abl[available].plot(kind='bar', ax=ax, colormap='Set2')
            ax.set_title('Ablation Study: Model Variant Comparison')
            ax.set_ylabel('Score')
            ax.set_ylim([0.5, 1.0])
            ax.legend(loc='lower right')
            plt.xticks(rotation=0)
            plt.tight_layout()
            plt.savefig('../results/figures/ablation_comparison.png', dpi=150, bbox_inches='tight')
            plt.show()
else:
    print('[INFO] Ablation results not yet available.')
    print('Train all variants: python train.py --variant lstm_only/cnn_only/hybrid')
    print('Then run: python scripts/evaluate.py --ablation')